# 09 Band Robustness

本课验证第 8 课的 `band_1pct` 是否具有初步稳健性。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from quant_trading.market_data import download_ohlcv
from quant_trading.strategy_improvement import (
    evaluate_band_cost_sensitivity,
    evaluate_band_sensitivity,
    plot_band_sensitivity,
)

## 1. 下载 SPY 数据

In [ ]:
df = download_ohlcv(symbol="SPY", start="2000-01-01", auto_adjust=True)
df.tail()

## 2. 参数敏感性

比较 0%、0.5%、1%、1.5%、2% 的均线差距过滤。

In [ ]:
band_pcts = [0.0, 0.005, 0.01, 0.015, 0.02]

results = evaluate_band_sensitivity(
    df,
    band_pcts=band_pcts,
    short_window=10,
    long_window=200,
    transaction_cost_bps=3.0,
    slippage_bps=2.0,
    commission_bps=1.0,
)
results

## 3. 样本外测试区

In [ ]:
results.loc[results["period"] == "test"][[
    "band_pct",
    "annualized_return",
    "max_drawdown",
    "calmar",
    "single_side_trades",
    "short_trade_contribution",
]]

## 4. 成本压力测试

In [ ]:
cost_results = evaluate_band_cost_sensitivity(
    df,
    band_pcts=[0.0, 0.01],
    cost_bps_values=[0, 3, 10, 25, 50],
    short_window=10,
    long_window=200,
)
cost_results

## 5. 图表

In [ ]:
plot_band_sensitivity(results);